In [94]:
import pandas as pd

In [95]:
df = pd.read_csv('100_Unique_QA_Dataset.csv')

In [96]:
df

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100
...,...,...
85,Who directed the movie 'Titanic'?,JamesCameron
86,Which superhero is also known as the Dark Knight?,Batman
87,What is the capital of Brazil?,Brasilia
88,Which fruit is known as the king of fruits?,Mango


In [97]:
# tokenize
def tokenize(text):
  text = text.lower()
  text = text.replace(',', '')
  text = text.replace('.', '')
  text = text.replace('?', '')
  text = text.replace("'","")
  return text.split()

In [98]:

tokenize("hello, how. 'are' you?")

['hello', 'how', 'are', 'you']

In [99]:
# voclabary
vocab = {'UKN':0}

In [100]:
# vocablary function where vocab is built
def build_vocab(row):

  tokenizes_question = tokenize(row['question'])
  tokenizes_answer = tokenize(row['answer'])

  merged_tokens = tokenizes_question + tokenizes_answer

  for token in merged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)

In [101]:
df.apply(build_vocab,axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [102]:

# unique words in our dataset
vocab

{'UKN': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'united

In [103]:
# conver words to numirical index
def text_to_indexes(text,vocab):

  indexed_text = []

  for token in tokenize(text):

    if token in vocab:
      indexed_text.append(vocab[token])

    else:
      indexed_text.append(vocab['UKN'])

  return indexed_text

In [104]:

text_to_indexes("what is onepiece",vocab)
# here we are getting indexies when text is given w.r.s to vocablary build above

[1, 2, 0]

In [105]:
import torch

from torch.utils.data import Dataset, DataLoader

In [106]:
from numpy._core import numeric
class QADataset(Dataset):

  def __init__(self,df,vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):

    numerical_question = text_to_indexes(self.df.iloc[index]['question'],self.vocab)

    numerical_answer = text_to_indexes(self.df.iloc[index]['answer'],self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [107]:
dataset = QADataset(df, vocab)

In [108]:
dataset[0]

(tensor([1, 2, 3, 4, 5, 6]), tensor([7]))

In [109]:
dataloader = DataLoader(dataset, batch_size = 1,shuffle=True)

In [110]:
for questions, answers in dataloader:
  print(questions,answers)

tensor([[42, 18,  2, 62, 63,  3, 64, 18]]) tensor([[65]])
tensor([[ 42, 318,   2,  62,  63,   3, 319,   5, 320]]) tensor([[321]])
tensor([[ 42, 255,   2, 256,  83, 257, 258]]) tensor([[259]])
tensor([[  1,  87, 229, 230, 231, 232]]) tensor([[233]])
tensor([[ 42,   2,   3, 210, 137, 168, 211, 169]]) tensor([[113]])
tensor([[ 1,  2,  3, 37, 38, 39, 40]]) tensor([[41]])
tensor([[ 42, 174,   2,  62,  39, 175, 176,  12, 177, 178]]) tensor([[179]])
tensor([[10, 29,  3, 30, 31]]) tensor([[32]])
tensor([[ 78,  79, 195,  81,  19,   3, 196, 197, 198]]) tensor([[199]])
tensor([[ 78,  79, 261, 151,  14, 262, 153]]) tensor([[36]])
tensor([[ 42,  86,  87, 241, 242,  19,  39, 243]]) tensor([[244]])
tensor([[10, 75, 76]]) tensor([[77]])
tensor([[ 42, 137, 118,   3, 247,   5, 248]]) tensor([[249]])
tensor([[  1,   2,   3,   4,   5, 135]]) tensor([[136]])
tensor([[  1,   2,   3,   4,   5, 109]]) tensor([[317]])
tensor([[ 78,  79, 150, 151,  14, 152, 153]]) tensor([[154]])
tensor([[ 42, 167,   2,   3,  1

In [111]:
import torch.nn as nn

In [131]:
class SimpleRnn(nn.Module):
  def __init__(self,voclab_Size):
    super().__init__()
    self.embedding = nn.Embedding(voclab_Size,embedding_dim=50)
    self.rnn = nn.RNN(50,64,batch_first = True)
    self.fc = nn.Linear(64,voclab_Size)

  def forward(self, input_tensor):
    embedded_questions = self.embedding(input_tensor)
    hidden, final = self.rnn(embedded_questions)
    output = self.fc(final.squeeze(0))

    return output

In [113]:
dataset[0]

(tensor([1, 2, 3, 4, 5, 6]), tensor([7]))

In [114]:
x =  nn.Embedding(324,embedding_dim=50)

In [115]:
a = x(dataset[0][0])
# here the embeddings is working fine, where there are 6 vectors of 50 dimentions each

In [116]:
a

tensor([[-1.0474e+00, -8.6015e-01, -1.4854e+00,  6.9924e-01, -1.0026e+00,
          1.8254e+00, -1.3294e-01,  3.4652e-01, -2.1206e+00, -1.4389e-01,
          9.3882e-01,  1.2054e+00, -1.1148e+00, -1.6716e+00, -1.2995e+00,
          1.0160e+00,  8.2815e-01,  7.2202e-01,  4.3125e-01,  1.5541e-01,
          1.1019e+00,  1.8470e-01, -2.9995e-01,  3.8346e-01, -3.0086e-01,
         -1.8549e-01,  1.1351e+00, -5.4901e-01, -5.6971e-01,  5.0376e-01,
          5.6203e-01,  2.5812e-01, -3.8660e-01, -6.6896e-01,  2.2210e-01,
         -9.2555e-01,  2.0176e+00,  7.2872e-02,  5.2016e-01, -2.9497e-01,
          1.6104e+00,  3.3195e-01,  1.3636e-01, -3.0833e-01,  1.3186e+00,
         -1.0556e+00,  1.4591e+00,  1.1216e+00, -2.4926e+00,  3.3130e-01],
        [ 1.0011e+00, -1.7723e+00, -1.1899e+00,  1.2692e+00,  1.4472e-01,
          7.2261e-01,  1.4988e+00,  1.0362e+00, -1.2375e+00, -1.0955e+00,
         -1.3922e+00, -1.7680e+00,  6.3831e-01, -1.1681e+00, -1.8573e+00,
         -2.5576e-01,  8.8187e-01, -8

In [117]:
x(dataset[0][0]).shape
# shape of our vector where there are 6 vectors of 50 dimentionals

torch.Size([6, 50])

In [118]:
y = nn.RNN(50,64)

In [119]:
y(a)

(tensor([[ 0.0267,  0.3677, -0.3188,  0.3641,  0.1059,  0.0343, -0.3426, -0.5761,
          -0.3074,  0.1160, -0.4312,  0.1129, -0.5735,  0.1420,  0.4463, -0.0086,
           0.0700, -0.3893,  0.3444,  0.0794,  0.0194,  0.2539,  0.0876, -0.0338,
          -0.1734,  0.5910,  0.1322,  0.3013, -0.5193, -0.1256,  0.2972,  0.4003,
           0.5470, -0.4756,  0.0631, -0.8436, -0.0671,  0.4881, -0.6129, -0.1772,
           0.3601, -0.2836, -0.0686, -0.2726,  0.2000, -0.2823, -0.5231, -0.4138,
           0.0771, -0.1257,  0.3184, -0.0741,  0.4675, -0.5926, -0.1125, -0.8481,
          -0.1240, -0.2984, -0.2211,  0.1337, -0.3716,  0.2883, -0.0432, -0.4107],
         [-0.4499,  0.2455,  0.2097, -0.3248,  0.5704, -0.0814, -0.3741,  0.0258,
           0.1596,  0.0715, -0.0597, -0.4457, -0.7355,  0.4356,  0.4014, -0.1296,
           0.4243, -0.0161,  0.7919,  0.3466, -0.1881,  0.2513, -0.7328,  0.6154,
          -0.5923, -0.3384, -0.7352,  0.6049,  0.1434,  0.5288,  0.3356,  0.3169,
          -0.64

In [120]:
y(a) # we got two tensors here

(tensor([[ 0.0267,  0.3677, -0.3188,  0.3641,  0.1059,  0.0343, -0.3426, -0.5761,
          -0.3074,  0.1160, -0.4312,  0.1129, -0.5735,  0.1420,  0.4463, -0.0086,
           0.0700, -0.3893,  0.3444,  0.0794,  0.0194,  0.2539,  0.0876, -0.0338,
          -0.1734,  0.5910,  0.1322,  0.3013, -0.5193, -0.1256,  0.2972,  0.4003,
           0.5470, -0.4756,  0.0631, -0.8436, -0.0671,  0.4881, -0.6129, -0.1772,
           0.3601, -0.2836, -0.0686, -0.2726,  0.2000, -0.2823, -0.5231, -0.4138,
           0.0771, -0.1257,  0.3184, -0.0741,  0.4675, -0.5926, -0.1125, -0.8481,
          -0.1240, -0.2984, -0.2211,  0.1337, -0.3716,  0.2883, -0.0432, -0.4107],
         [-0.4499,  0.2455,  0.2097, -0.3248,  0.5704, -0.0814, -0.3741,  0.0258,
           0.1596,  0.0715, -0.0597, -0.4457, -0.7355,  0.4356,  0.4014, -0.1296,
           0.4243, -0.0161,  0.7919,  0.3466, -0.1881,  0.2513, -0.7328,  0.6154,
          -0.5923, -0.3384, -0.7352,  0.6049,  0.1434,  0.5288,  0.3356,  0.3169,
          -0.64

In [121]:
y(a)[0].shape # hidden state

torch.Size([6, 64])

In [122]:
b = y(a)[1] # final state

In [123]:
z = nn.Linear(64,324)

In [124]:
z(b)

tensor([[-0.5438,  0.3801,  0.2492,  0.0571, -0.3371, -0.0391,  0.1060, -0.2393,
         -0.3935,  0.1648, -0.0670,  0.1826, -0.3943,  0.3538,  0.2407,  0.2420,
         -0.4088,  0.4704, -0.5160, -0.4404,  0.1385,  0.3200,  0.1979, -0.6650,
         -0.1650,  0.5849,  0.6746, -0.4023, -0.2149,  0.8844, -0.1256, -0.4666,
         -0.0542, -0.3217,  0.0106, -0.1562, -0.1979,  0.3570, -0.2450, -0.1131,
          0.4357, -0.0503,  0.1315, -0.1787, -0.4391,  0.0118,  0.5101,  0.3588,
          0.2334,  0.1319, -0.2023,  0.1668, -0.3529,  0.2832,  0.2084,  0.2424,
         -0.0987,  0.4355, -0.1982,  0.0630, -0.0732,  0.3883,  0.0062,  0.1346,
          0.3994,  0.4786, -0.0183,  0.5269, -0.3535, -0.1605, -0.1925, -0.2859,
         -0.5506, -0.2004, -0.2557, -0.5377,  0.1865, -0.1751,  0.0396,  0.2997,
          0.0228,  0.1594,  0.0817,  0.2346, -0.2104, -0.1103,  0.2213, -0.8196,
         -0.3992, -0.0211, -0.2081, -0.1330, -0.4751,  0.2462,  0.1140,  0.1863,
          0.1921, -0.0021, -

In [125]:
z(b).shape # for each word in vocalabry we are getting probablity

torch.Size([1, 324])

In [126]:
learning_rate = 0.001
epochs = 20

In [132]:
model = SimpleRnn(len(vocab))

In [133]:
criterion =  nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)

In [135]:
# training

for echos in range(epochs):
  total_loss = 0

  for questions, answers in dataloader:
    optimizer.zero_grad()

    # forward pass
    output = model(questions)

    # loss
    loss = criterion(output, answers.squeeze(1))

    # gredident claculate
    loss.backward()

    # update gradidents
    optimizer.step()

    total_loss = total_loss + loss.item()

  # The original code had a typo `epoch` instead of `echos`.
  print(f"epoch {echos + 1},Loss {total_loss:.4f}")


epoch 1,Loss 525.2455
epoch 2,Loss 460.1332
epoch 3,Loss 383.0451
epoch 4,Loss 319.4938
epoch 5,Loss 267.1983
epoch 6,Loss 219.1135
epoch 7,Loss 174.7487
epoch 8,Loss 135.9678
epoch 9,Loss 103.9639
epoch 10,Loss 79.0536
epoch 11,Loss 60.6482
epoch 12,Loss 47.0214
epoch 13,Loss 37.2547
epoch 14,Loss 29.9018
epoch 15,Loss 24.6631
epoch 16,Loss 20.4687
epoch 17,Loss 17.3311
epoch 18,Loss 14.7279
epoch 19,Loss 12.7605
epoch 20,Loss 10.9383


In [148]:
def predict(model, question, treshold=0.5):
  # convert question to numbers
  numerical_question = text_to_indexes(question, vocab)

  #convert num to rensors
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)
  #send to model
  output = model(question_tensor)

  #convert logits to probs
  probs =torch.nn.functional.softmax(output, dim=1)

  #find index of max prob
  value, index = torch.max(probs, dim=1)

  # check treshold
  if value < treshold:
    print('i dont know..!')

  else:
    print(list(vocab.keys())[index])

In [149]:
predict(model, "what is onepiece")

i dont know..!


In [150]:
predict(model, "what is the capital of france")

paris


In [151]:
predict(model, "what is the largest planet of our solar system")

jupiter


In [152]:
predict(model, "say my name")

i dont know..!
